In [1]:
import os
import sys
import gc
from argparse import ArgumentParser, Namespace

import math
import cmath
import random
import time

from tqdm import tqdm, trange
from typing import List, Tuple, Dict, Set, Union, Optional, Literal, Any, Iterable, Callable, NamedTuple
from dataclasses import dataclass, field

import numpy as np
from numpy import array, ndarray, linalg, matrix, strings, testing
import numba
from numba.typed import List as NumbaList, Dict as NumbaDict
from numba import types
import scipy.sparse

from sim_hash import SimHash, test_simhash

# Type aliases matching the reference
Array0D = ndarray
Array1D = ndarray
Array2D = ndarray
Array3D = ndarray

# Define Numba types globally to avoid inference errors inside njit
nb_int64 = types.int64
nb_int32 = types.int32
nb_list_int64 = types.ListType(types.int64)
nb_list_int32 = types.ListType(types.int32)

# Define types for CSR-like inverted index
nb_int32_arr = types.int32[:]
# (hashes, offsets, indices)
nb_table_struct = types.Tuple((nb_int32_arr, nb_int32_arr, nb_int32_arr))
nb_pool_struct = types.ListType(nb_table_struct)
nb_index_struct = types.ListType(nb_pool_struct)

CURRENT_PATH: str = os.getcwd()
REPO_PATH: str = os.path.dirname(CURRENT_PATH)
DATASET_DIR: str = os.path.join(REPO_PATH, "data")

In [2]:
@numba.njit(parallel=True)
def create_random_pools(
        num_features: int,
        num_pools: int, 
        points_per_pool: int,
        pools_per_point: int
) -> Tuple[Array2D, Array2D]:
    """
    Create random pool assignment where each point appears in exactly pools_per_point pools.
    Uses a greedy randomized approach to minimize pool overlaps.
    """
    # Validate parameters
    if points_per_pool * num_pools < pools_per_point * num_features:
        # Numba doesn't support f-strings in raise
        raise ValueError("Inconsistent pooling parameters: Capacity < Demand")
    
    pools = np.full((num_pools, points_per_pool), -1, dtype=np.int64)
    point_pool_count = np.zeros(num_features, dtype=np.int64)
    pooling_matrix = np.zeros((num_pools, num_features), dtype=np.bool_)
    
    # Create assignments array directly
    total_assignments = num_features * pools_per_point
    point_assignments = np.empty(total_assignments, dtype=np.int64)
    
    curr_idx = 0
    for point_idx in range(num_features):
        for _ in range(pools_per_point):
            point_assignments[curr_idx] = point_idx
            curr_idx += 1
    
    print("Creating random pool assignment...")
    np.random.shuffle(point_assignments)
    pool_fill_count = np.zeros(num_pools, dtype=np.int64)

    for idx in range(len(point_assignments)):
        point_idx = point_assignments[idx]
        # Try random pools first
        selected_pool = -1
        for _ in range(100):
            p = np.random.randint(0, num_pools)
            if pool_fill_count[p] < points_per_pool and not pooling_matrix[p, point_idx]:
                selected_pool = p
                break
        if selected_pool == -1:
            # Fallback: linear search
            candidates = np.where((pool_fill_count < points_per_pool) & (~pooling_matrix[:, point_idx]))[0]
            if len(candidates) == 0:
                raise ValueError("Cannot assign point to pool (capacity exceeded)")
            selected_pool = candidates[np.random.randint(0, len(candidates))]

        pools[selected_pool, pool_fill_count[selected_pool]] = point_idx
        pooling_matrix[selected_pool, point_idx] = True
        pool_fill_count[selected_pool] += 1
        point_pool_count[point_idx] += 1
        if idx % 100000 == 0:
            print("Assigned", idx, "points")
    return pools, pooling_matrix

In [3]:
@numba.njit
def build_inverted_index(
    hash_features: Array2D,
    pools: Array2D
) -> NumbaList:
    """
    Build inverted index for hash features within each pool.
    Returns a NumbaList of NumbaLists of Tuples (CSR-like structure).
    Outer: pools, Middle: tables, Inner: (hashes, offsets, indices).
    """
    num_pools, points_per_pool = pools.shape
    num_tables = hash_features.shape[1]
    
    # Outer list: List of pools
    inverted_hash_tables = NumbaList()
    
    print("Building per-pool inverted index (CSR optimized)...")
    for pool_idx in range(num_pools):
        if pool_idx % 500 == 0:
            print("Built", pool_idx, "pool indices")
        # Inner list: List of tables for this pool
        pool_inverted_tables = NumbaList()
        pool_points = pools[pool_idx]
        
        for table_idx in range(num_tables):
            # Collect valid points
            valid_indices = []
            for p in pool_points:
                if p != -1:
                    valid_indices.append(p)
            
            if len(valid_indices) > 0:
                hashes = np.empty(len(valid_indices), dtype=np.int32)
                points = np.empty(len(valid_indices), dtype=np.int32)
                
                for i, p in enumerate(valid_indices):
                    hashes[i] = hash_features[p, table_idx]
                    points[i] = p
                
                # Sort by hash
                sort_idx = np.argsort(hashes)
                sorted_hashes = hashes[sort_idx]
                sorted_points = points[sort_idx]
                
                # Manual unique and offsets (np.unique return_index not supported in Numba)
                num_unique = 1
                for i in range(1, len(sorted_hashes)):
                    if sorted_hashes[i] != sorted_hashes[i-1]:
                        num_unique += 1
                
                hashes_arr = np.empty(num_unique, dtype=np.int32)
                offsets_arr = np.empty(num_unique + 1, dtype=np.int32)
                indices_arr = sorted_points
                
                curr_h = sorted_hashes[0]
                hashes_arr[0] = curr_h
                offsets_arr[0] = 0
                
                u_idx = 1
                for i in range(1, len(sorted_hashes)):
                    h = sorted_hashes[i]
                    if h != curr_h:
                        offsets_arr[u_idx] = i
                        hashes_arr[u_idx] = h
                        curr_h = h
                        u_idx += 1
                offsets_arr[u_idx] = len(sorted_hashes)
                
                # Append tuple
                pool_inverted_tables.append((hashes_arr, offsets_arr, indices_arr))
            else:
                # Empty table
                empty_arr = np.empty(0, dtype=np.int32)
                pool_inverted_tables.append((empty_arr, np.array([0], dtype=np.int32), empty_arr))
            
        inverted_hash_tables.append(pool_inverted_tables)
        
    return inverted_hash_tables

In [4]:
@numba.njit
def popcount(n: int) -> int:
    """
    Count the number of set bits in an integer.
    """
    c = 0
    while n > 0:
        c += 1
        n &= n - 1
    return c

In [5]:
@numba.njit
def does_pool_match_query_hash(
    query_hashes: Array1D,
    inverted_index: NumbaList,
    hash_threshold: int,
    match_threshold: int
) -> bool:
    """
    Check if a pool matches the query hash using the per-pool inverted index (CSR).
    """
    # Use a dictionary for sparse counting since point indices are global and large
    # Key: point_idx, Value: count
    match_counts = NumbaDict.empty(nb_int32, nb_int32)
    
    num_tables = len(query_hashes)
    
    for table_idx in range(num_tables):
        query_hash = query_hashes[table_idx]
        # Unpack tuple
        hashes, offsets, indices = inverted_index[table_idx]
        
        if len(hashes) == 0:
            continue
            
        if hash_threshold == 0:
            # Binary search
            idx = np.searchsorted(hashes, query_hash)
            if idx < len(hashes) and hashes[idx] == query_hash:
                start = offsets[idx]
                end = offsets[idx+1]
                for i in range(start, end):
                    p = indices[i]
                    c = match_counts.get(p, nb_int32(0)) + nb_int32(1)
                    match_counts[p] = c
                    if c >= match_threshold:
                        return True
        else:
            # Iterate all hashes
            for i in range(len(hashes)):
                h_val = hashes[i]
                # Hamming distance
                xor_val = h_val ^ query_hash
                dist = 0
                while xor_val > 0:
                    dist += 1
                    xor_val &= xor_val - 1
                
                if dist <= hash_threshold:
                    start = offsets[i]
                    end = offsets[i+1]
                    for k in range(start, end):
                        p = indices[k]
                        c = match_counts.get(p, nb_int32(0)) + nb_int32(1)
                        match_counts[p] = c
                        if c >= match_threshold:
                            return True
                    
    return False

In [6]:
@numba.njit(parallel=True)
def find_matching_points(
    hash_features: Array2D,
    query_hashes: Array1D,
    hash_threshold: int,
    match_threshold: int
) -> List[int]:
    """
    Find all points that match the query hash criteria.
    """
    num_points, num_tables = hash_features.shape
    matching_indices = []
    
    for i in range(num_points):
        matches = 0
        for t in range(num_tables):
            h_feat = hash_features[i, t]
            h_query = query_hashes[t]
            
            # Compute Hamming distance
            xor_val = h_feat ^ h_query
            dist = 0
            while xor_val > 0:
                dist += 1
                xor_val &= xor_val - 1
            
            if dist <= hash_threshold:
                matches += 1
        
        if matches >= match_threshold:
            matching_indices.append(i)
            
    matching_indices.sort()
    return matching_indices

In [7]:
def my_matching_algo(
    pooling_matrix: ndarray,
    positive_pools: ndarray
) -> ndarray:
    """
    My implementation of matching algorithm, to solve for sparse vector x given y = Ax, 
        where y is the positive_pools and A is the pooling_matrix.
    
    :param pooling_matrix: The pooling matrix A
    :type pooling_matrix: ndarray
    :param positive_pools: The output vector y
    :type positive_pools: ndarray
    :return: The recovered sparse vector x
    :rtype: ndarray
    """
    num_pools, num_features = pooling_matrix.shape
    assert positive_pools.shape == (num_pools,), "Incompatible shapes between pooling matrix and positive pools"
    possible = np.ones(num_features, dtype=bool)
    candidates = np.zeros(num_features, dtype=bool)
    print(f"[my_matching_algo] Initial possible.sum(): {possible.sum()} (should be {num_features})")
    # Eliminate features present in any negative pool
    for pool_idx in range(num_pools):
        if positive_pools[pool_idx] == 0:
            before = possible.sum()
            possible &= ~pooling_matrix[pool_idx]
            after = possible.sum()
    print(f"[my_matching_algo] After negative pool elimination, possible.sum(): {possible.sum()}")
    iter_count = 0
    while True:
        newly_identified = np.zeros(num_features, dtype=bool)
        for pool_idx in range(num_pools):
            if not positive_pools[pool_idx]:
                continue
            pool_points = pooling_matrix[pool_idx] & possible
            if np.sum(pool_points) == 1:
                newly_identified |= pool_points
        
        if not np.any(newly_identified):
            print(f"[my_matching_algo] Iter {iter_count}: No new identifications, breaking loop.")
            break
        print(f"[my_matching_algo] Iter {iter_count}: Identified {newly_identified.sum()} new candidates.")
        candidates |= newly_identified
        possible &= ~newly_identified
        identified_pools = pooling_matrix @ newly_identified
        positive_pools &= ~identified_pools
        print(f"[my_matching_algo] Iter {iter_count}: Remaining possible.sum(): {possible.sum()}, Remaining positive_pools.sum(): {positive_pools.sum()}")
        iter_count += 1
    print(f"[my_matching_algo] Final candidates.sum(): {candidates.sum()}")
    return candidates

In [8]:
@numba.njit(parallel=True)
def OMP(
    A: ndarray,
    y: ndarray,
    s: int = 100,
) -> ndarray:
    """
    Solves the orthogonal matching persuit (OMP) problem for sparse x, y = Ax.

    Args:
    - A: The pooling matrix (m x n)
    - y: The output vector (the pool tests, m)
    - s: Maximum sparsity of x
    """
    m, n = A.shape
    y = y.astype(np.float64)
    x = np.zeros(n, dtype=np.float64)
    residual = y.copy()
    support = np.zeros(s, dtype=np.int64)
    support_size = 0
    
    for iteration in range(s):
        if np.linalg.norm(residual) < 1e-10:
            break
        correlations = A.T @ residual
        best_idx = np.argmax(np.abs(correlations))
        
        # Check if already in support
        already_in = False
        for i in range(support_size):
            if support[i] == best_idx:
                already_in = True
                break
        if already_in:
            break
            
        support[support_size] = best_idx
        support_size += 1
        
        # Solve least squares: (A_s^T A_s) x_s = A_s^T y using normal equations
        A_s = A[:, support[:support_size]]
        ATA = A_s.T @ A_s
        ATy = A_s.T @ y
        x_s = np.linalg.solve(ATA, ATy)
        
        # Update residual
        residual = y - A_s @ x_s
    
    # Build sparse solution
    for i in range(support_size):
        A_s = A[:, support[:support_size]]
        ATA = A_s.T @ A_s
        ATy = A_s.T @ y
        x_s = np.linalg.solve(ATA, ATy)
        for j in range(support_size):
            x[support[j]] = x_s[j]
        break
    
    return x

In [9]:
class MLGT:
    """
    Multi-Level Group Testing (MLGT): Efficient approximate nearest neighbor search
    combining group testing theory with locality-sensitive hashing.
    
    Adapted to use the imported SimHash class.
    """
    def __init__(
            self,
            num_tables: int = 500,
            hash_bits: int = 10,
            input_dim: int = 1000,
            hash_threshold: int = 2,
            match_threshold: int = 20,
            num_pools: int = 5000,
            pools_per_point: int = 3,
            points_per_pool: int = 600,
            features: Optional[Array2D] = None,
            features_path: Optional[str] = None,
            use_srp: bool = False,
            threshold: Optional[int] = None,
    ) -> None:
        """
        Initialize the MLGT instance with parameters for the SimHash index.

        Args:
            num_tables (int): Number of hash tables.
            hash_bits (int): Number of bits per hash.
            input_dim (int): Dimension of input vectors.
            hash_threshold (int): Hamming distance threshold for hash matching.
            match_threshold (int): Minimum number of table matches for a point to be considered a candidate.
            num_pools (int): Number of pools for group testing.
            pools_per_point (int): Number of pools each point is assigned to.
            points_per_pool (int): Number of points per pool.
            features (Optional[Array2D]): Feature matrix (N, D).
            features_path (Optional[str]): Path to feature matrix file.
            use_srp (bool): Whether to use Sparse Random Projections (not implemented).
            threshold (Optional[int]): Alias for hash_threshold.
        """
        # Validate parameters
        self.num_tables = num_tables
        self.hash_bits = hash_bits
        self.input_dim = input_dim
        # support alias `threshold` for backwards compatibility
        self.hash_threshold = threshold if threshold is not None else hash_threshold
        self.match_threshold = match_threshold
        self.num_pools = num_pools
        self.pools_per_point = pools_per_point
        self.points_per_pool = points_per_pool
        self.use_srp = use_srp
        
        # Pooling capacity check deferred to build_index (dataset-dependent)

        # Load features if provided
        self.features: Optional[Array2D] = features
        self.num_features: int = 0
        if self.features is not None:
            assert self.features.ndim == 2, "Features must be a 2D array."
            assert self.features.shape[1] == input_dim, f"Features must have dimension {input_dim}, got {self.features.shape[1]}."
            self.num_features = self.features.shape[0]
        elif features_path is not None:
            if not os.path.exists(features_path):
                raise ValueError(f"Features path {features_path} does not exist.")
            self.features = np.load(features_path)
            assert self.features.ndim == 2, "Features must be a 2D array."
            assert self.features.shape[1] == input_dim, f"Features must have dimension {input_dim}, got {self.features.shape[1]}."
            self.num_features = self.features.shape[0]
        else:
            raise ValueError("Atleast one of `features` or `features_path` must be provided.")
        
        # Initialize other attributes
        self.hash_function: Optional[SimHash] = None
        self.hash_features: Optional[Array2D] = None
        self.pools: Optional[Array2D] = None
        self.pooling_matrix: Optional[Array2D] = None
        self.inverted_hash_tables: NumbaList = NumbaList()

        # Initialize the SimHash index
        self.hash_function = SimHash(
            num_hashes=num_tables, 
            num_bits=hash_bits, 
            threshold=self.hash_threshold, 
            dimension=input_dim
        )
        
        if self.hash_threshold >= self.hash_bits:
            print(f"WARNING: hash_threshold ({self.hash_threshold}) >= hash_bits ({self.hash_bits}). "
                  "This will cause all points to match, leading to slow queries and poor recall.")


    
    def build_index(self) -> None:
        """
        Build the index for the dataset using the SimHash class.
        """
        # Ensure capacity is sufficient
        required_capacity = self.pools_per_point * self.num_features
        current_capacity = self.num_pools * self.points_per_pool
        if current_capacity < required_capacity:
            new_ppp = math.ceil(required_capacity / self.num_pools) + 10 # Add buffer
            print(f"Warning: Adjusting points_per_pool from {self.points_per_pool} to {new_ppp} to satisfy demand.")
            self.points_per_pool = int(new_ppp)

        # Use pure Python version with tqdm (Numba removed)
        self.pools, self.pooling_matrix = create_random_pools(
            num_features=self.num_features,
            num_pools=self.num_pools,
            points_per_pool=self.points_per_pool,
            pools_per_point=self.pools_per_point
        )
        print(f"DEBUG: Pools shape: {self.pools.shape}, Pooling matrix shape: {self.pooling_matrix.shape}")
        
        # Optimization: Use int32 for hash features to save memory (sufficient for hash values)
        self.hash_features: Array2D = self.hash_function.hash_bits_to_value(self.hash_function(self.features)).astype(np.int32)
        print(f"DEBUG: Hash features shape: {self.hash_features.shape}")
        gc.collect()

        print(f"DEBUG: Computing inverted hash tables...")
        self.inverted_hash_tables = build_inverted_index(
            hash_features=self.hash_features,
            pools=self.pools
        )

        # Integrity checks / assertions: ensure pooling matrix dimensions and counts are reasonable
        assert self.pooling_matrix.shape == (self.num_pools, self.num_features), (
            f"pooling_matrix shape mismatch: expected ({self.num_pools}, {self.num_features}), got {self.pooling_matrix.shape}"
        )
        # Each point should appear in approximately `pools_per_point` pools (allow off-by-one due to ceil)
        per_point_counts = np.sum(self.pooling_matrix, axis=0)
        if self.pools_per_point is not None:
            min_expected = max(0, self.pools_per_point - 1)
            max_expected = self.pools_per_point + 1
            assert np.all((per_point_counts >= min_expected) & (per_point_counts <= max_expected)), (
                f"Per-point pool counts outside expected range [{min_expected},{max_expected}] (example counts: {per_point_counts[:10]})"
            )
        # Each pool should contain approximately `points_per_pool` points
        per_pool_counts = np.sum(self.pooling_matrix, axis=1)
        expected_avg = (self.num_features * self.pools_per_point) / self.num_pools
        # Allow some variance, especially if points_per_pool was padded
        min_pool = int(expected_avg * 0.8)
        max_pool = self.points_per_pool
        assert np.all((per_pool_counts >= min_pool) & (per_pool_counts <= max_pool)), (
            f"Per-pool point counts outside expected range [{min_pool},{max_pool}] (example counts: {per_pool_counts[:10]})"
        )
    
    def _solve_scipy(self, positive_pools: ndarray, method: str = "lsmr") -> List[int]:
        """
        Solve the group testing problem using SciPy's sparse solvers (LSMR or LSQR).
        """
        from scipy.sparse import csr_matrix
        
        # Avoid creating a dense float matrix (which would be huge)
        # Create CSR directly from boolean matrix, specifying float dtype
        A_sparse = csr_matrix(self.pooling_matrix, dtype=float)
        y = positive_pools.astype(float)
        
        if method == "lsmr":
            from scipy.sparse.linalg import lsmr
            sol = lsmr(A_sparse, y)
            x = sol[0]
        elif method == "lsqr":
            from scipy.sparse.linalg import lsqr
            sol = lsqr(A_sparse, y)
            x = sol[0]
        else:
            raise ValueError(f"Unknown method: {method}")
            
        # Post-process solution: clamp negatives, handle NaNs
        x = np.nan_to_num(x)
        
        # Candidates from solver: threshold at 0.5 (features with solution > 0.5)
        solver_positives = np.where(x > 0.5)[0].tolist()

        # If solver yields no positives, take top few entries from x (smallest non-zero set)
        if len(solver_positives) == 0:
            # pick top 1% features or at least 1
            k = max(1, int(0.01 * self.num_features))
            solver_positives = np.argsort(-x)[:k].tolist()

        return sorted(list(set(solver_positives)))
    
    def _solve_omp(self, positive_pools: ndarray, sparsity: int = 100) -> List[int]:
        """
        Solve the group testing problem using Orthogonal Matching Pursuit (OMP).
        """
        A = self.pooling_matrix.astype(float)
        y = positive_pools.astype(float)
        
        x = OMP(A, y, s=sparsity)
        x = np.nan_to_num(x)
        
        # Threshold at 0.5
        omp_positives = np.where(x > 0.5)[0].tolist()
        
        # Fallback: take top candidates if none found
        if len(omp_positives) == 0:
            k = max(1, int(0.01 * self.num_features))
            omp_positives = np.argsort(-x)[:k].tolist()
        
        return sorted(list(set(omp_positives)))

    def query(
            self, 
            query_vector: Array1D,
            algorithm: Literal["my_algo", "saffron", "grotesque", "lsmr", "lsqr"] = "my_algo"
    ) -> List[int]:
        """
        Query the MLGT index with a given query vector.
        """
        # print(f"DEBUG: Querying with hash_threshold={self.hash_threshold}, match_threshold={self.match_threshold}, hash_bits={self.hash_bits}")
        
        positive_pools: ndarray = np.zeros((self.num_pools,), dtype=bool)
        query_hash = self.hash_function(query_vector)
        # Optimization: Ensure query hash values are int32 to match inverted index keys
        query_hash_values = self.hash_function.hash_bits_to_value(query_hash).astype(np.int32)

        for pool_idx in tqdm(range(self.num_pools), desc="Querying pools...", leave=False):
            positive_pools[pool_idx] = does_pool_match_query_hash(
                query_hashes=query_hash_values,
                inverted_index=self.inverted_hash_tables[pool_idx],
                hash_threshold=self.hash_threshold,
                match_threshold=self.match_threshold
            )
        
        # DEBUG: Verify positive pools against ground truth (brute-force check)
        if self.hash_threshold == 0:
            true_hash_matches: ndarray = ((self.hash_features == query_hash_values).astype(np.int32).sum(axis=1) >= self.match_threshold).astype(bool)
            true_positive_pools: ndarray = (self.pooling_matrix @ true_hash_matches).astype(bool)
            
            mismatches = np.sum(positive_pools != true_positive_pools)
            if mismatches > 0:
                print(f"DEBUG: Positive pool mismatch! {mismatches} pools differ.")
                print(f"DEBUG: Computed positives: {np.sum(positive_pools)}, True positives: {np.sum(true_positive_pools)}")
                # assert np.all(positive_pools == true_positive_pools), "DEBUG: Positive pool computation mismatch!"
        
        candidates: List[int] = []
        if algorithm == "my_algo":
            candidates_mask = my_matching_algo(
                pooling_matrix=self.pooling_matrix,
                positive_pools=positive_pools
            )
            candidates = np.where(candidates_mask)[0].tolist()
        elif algorithm == "omp":
            candidates = self._solve_omp(positive_pools)
        elif algorithm in ["lsmr", "lsqr"]:
            candidates = self._solve_scipy(positive_pools, method=algorithm)
        elif algorithm == "saffron":
            raise NotImplementedError("Saffron algorithm not implemented in this version.")
        elif algorithm == "grotesque":
            raise NotImplementedError("GROTESQUE algorithm not implemented in this version.")
        else:
            raise ValueError(f"Unknown algorithm: {algorithm}")

        n_cand = len(candidates)
        if n_cand == 0:
            # print(f"[{algorithm}] No candidates found.")
            pass

        return candidates
    
    def __call__(self, query_vector: Array1D) -> List[int]:
        return self.query(query_vector)

    def get_hash_neighbors(self, query_vector: Array1D) -> List[int]:
        """
        Get all neighbors that satisfy the hash matching criteria (threshold).
        """
        query_hash = self.hash_function(query_vector)
        query_hash_values = self.hash_function.hash_bits_to_value(query_hash)
        
        return find_matching_points(
            self.hash_features,
            query_hash_values,
            self.hash_threshold,
            self.match_threshold
        )

    def precision_and_recall(
        self, 
        query_vector: Array1D, 
        mlgt_neighbors: List[int], 
        brute_force_neighbors: List[int], 
        num_neighbors: int = 100
    ) -> Tuple[float, float, float]:
        """
        Compute precision, recall, and F1 score.
        """
        s_mlgt = set(mlgt_neighbors)
        s_true = set(brute_force_neighbors)
        
        intersection = len(s_mlgt.intersection(s_true))
        
        precision = intersection / len(s_mlgt) if len(s_mlgt) > 0 else 1.0
        recall = intersection / len(s_true) if len(s_true) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        return precision, recall, f1
    
    def brute_force_search(self, query_vector: Array1D, num_neighbors: int = 100) -> List[int]:
        """
        Perform brute-force search to find the top-k nearest neighbors.
        """
        similarities = self.features @ query_vector / (linalg.norm(self.features, axis=1) * linalg.norm(query_vector) + 1e-10)
        topk_indices = np.argsort(-similarities)[:num_neighbors]
        return topk_indices.tolist()
    
    def get_metrics(
            self, 
            query_vector: Array1D, 
            algorithm: str = "my_algo"
    ) -> Tuple[float, float, float, float, float, float, float]:
        """
        Get precision (final), recall (final), recall (search vs hash-matches), recall (hash-matches vs true) and times
        """
        query_start = time.time()
        saffron_indices = self.query(query_vector, algorithm=algorithm)
        query_end = time.time()

        hash_start = time.time()
        # Get candidates that satisfy BOTH thresholds (hash_threshold and match_threshold)
        # This is the "Ground Truth" for the Group Testing problem
        hash_threshold_neighbors = self.get_hash_neighbors(query_vector)
        hash_end = time.time()

        true_start = time.time()
        true_topk = self.brute_force_search(query_vector, num_neighbors=100)
        true_end = time.time()

        # Metrics
        set_saff = set(saffron_indices)
        set_true = set(true_topk)
        set_hash_thresh = set(hash_threshold_neighbors)

        # 1. Precision: How many of Saffron's output are in True Top 100?
        final_precision = len(set_saff.intersection(set_true)) / max(len(set_saff), 1)
        
        # 2. Recall: How many of True Top 100 did Saffron find?
        final_recall = len(set_saff.intersection(set_true)) / 100.0
        
        # 3. Recall (Saffron vs Hash Thresholds): Did Saffron find the points that satisfy the thresholds?
        # This measures the quality of the Group Testing recovery (Saffron)
        denom_saff_hash = len(set_hash_thresh)
        recall_search_vs_hash = len(set_saff.intersection(set_hash_thresh)) / denom_saff_hash if denom_saff_hash > 0 else 1.0
        precision_search_vs_hash = len(set_saff.intersection(set_hash_thresh)) / max(len(set_saff), 1)

        # 4. Recall (Hash Thresholds vs True): Do the points satisfying thresholds contain the True Top 100?
        # This measures the quality of the LSH parameters (tables, bits, thresholds)
        recall_hash_vs_true = len(set_hash_thresh.intersection(set_true)) / 100.0
        precision_hash_vs_true = len(set_hash_thresh.intersection(set_true)) / max(len(set_hash_thresh), 1)

        # 5. F1 for Saffron vs True
        f1 = 2 * (final_precision * final_recall) / (final_precision + final_recall) if (final_precision + final_recall) > 0 else 0.0

        # --- Debug prints per query ---
        print("-- Hash vs True (LSH quality) --"
              f" Precision: {precision_hash_vs_true:.4f},"
              f" Recall: {recall_hash_vs_true:.4f},"
              f" Sizes: |hash|={len(set_hash_thresh)}, |true|={len(set_true)}, |∩|={len(set_hash_thresh.intersection(set_true))}")

        print("-- MLGT vs Hash (Recovery quality) --"
              f" Precision: {precision_search_vs_hash:.4f},"
              f" Recall: {recall_search_vs_hash:.4f},"
              f" Sizes: |mlgt|={len(set_saff)}, |hash|={len(set_hash_thresh)}, |∩|={len(set_saff.intersection(set_hash_thresh))}")

        print("-- MLGT vs True (End-to-end) --"
              f" Precision: {final_precision:.4f},"
              f" Recall: {final_recall:.4f},"
              f" F1: {f1:.4f},"
              f" Sizes: |mlgt|={len(set_saff)}, |true|={len(set_true)}, |∩|={len(set_saff.intersection(set_true))}")

        return final_precision, final_recall, recall_search_vs_hash, recall_hash_vs_true, query_end - query_start, hash_end - hash_start, true_end - true_start



In [10]:
num_tables: int = 500
hash_bits: int = 10
input_dim: int = 1000
hash_threshold: int = 0
match_threshold: int = 50
num_queries: int = 100
num_pools: int = 5000
pools_per_point: int = 10
algorithm: Literal["my_algo", "saffron", "grotesque", "lsmr", "lsqr"] = "my_algo"
dataset: str = "imagenet"

dataset_path: str = os.path.join(DATASET_DIR, dataset)
if not os.path.exists(dataset_path):
    raise ValueError(f"Dataset path {dataset_path} does not exist. Please download the dataset first.")

In [11]:
mlgt = MLGT(
    num_tables=num_tables,
    hash_bits=hash_bits,
    input_dim=input_dim,
    hash_threshold=hash_threshold,
    match_threshold=match_threshold,
    num_pools=num_pools,
    pools_per_point=pools_per_point,
    features_path=os.path.join(dataset_path, "X.npy")
)
mlgt.build_index()

Creating random pool assignment...
Assigned 0 points
Assigned 100000 points
Assigned 200000 points
Assigned 300000 points
Assigned 400000 points
Assigned 500000 points
Assigned 600000 points
Assigned 700000 points
Assigned 800000 points
Assigned 900000 points
Assigned 1000000 points
Assigned 1100000 points
Assigned 1200000 points
Assigned 1300000 points
Assigned 1400000 points
Assigned 1500000 points
Assigned 1600000 points
Assigned 1700000 points
Assigned 1800000 points
Assigned 1900000 points
Assigned 2000000 points
Assigned 2100000 points
Assigned 2200000 points
Assigned 2300000 points
Assigned 2400000 points
Assigned 2500000 points
Assigned 2600000 points
Assigned 2700000 points
Assigned 2800000 points
Assigned 2900000 points
Assigned 3000000 points
Assigned 3100000 points
Assigned 3200000 points
Assigned 3300000 points
Assigned 3400000 points
Assigned 3500000 points
Assigned 3600000 points
Assigned 3700000 points
Assigned 3800000 points
Assigned 3900000 points
Assigned 4000000 poi

In [12]:
query_set: Array2D = np.load(os.path.join(dataset_path, "Q.npy"))

In [ ]:
dataset_hashes: ndarray = mlgt.hash_features
query_hashes: ndarray = mlgt.hash_function.hash_bits_to_value(mlgt.hash_function(query_set)).astype(np.int32)
pooling_matrix: ndarray = mlgt.pooling_matrix
assert dataset_hashes.shape == (mlgt.num_features, mlgt.num_tables)
assert query_hashes.shape[1] == mlgt.num_tables
print(mlgt.pools.shape)

# Metrics for all algorithms
K = 100  # Top K to retrieve
metrics = {
    "my_algo": {"precision_vs_hash": [], "recall_vs_hash": [], "precision_vs_true": [], "recall_vs_true": []},
    "lsmr": {"precision_vs_hash": [], "recall_vs_hash": [], "precision_vs_true": [], "recall_vs_true": []},
    "omp": {"precision_vs_hash": [], "recall_vs_hash": [], "precision_vs_true": [], "recall_vs_true": []}
}

for qidx in tqdm(range(min(10, query_set.shape[0])), desc="Validating queries..."):
    # Hash matches (group testing ground truth)
    hash_match_scores = (dataset_hashes == query_hashes[qidx]).astype(np.int32).sum(axis=1)
    topK_hash_indices = np.argsort(-hash_match_scores)[:K]
    
    # True nearest neighbors (brute force dot product)
    query_vec = query_set[qidx]
    similarities = mlgt.features @ query_vec / (np.linalg.norm(mlgt.features, axis=1) * np.linalg.norm(query_vec) + 1e-10)
    topK_true_indices = np.argsort(-similarities)[:K]
    
    # Positive pools for group testing
    true_matches: ndarray = hash_match_scores >= mlgt.match_threshold
    positive_pools: ndarray = (pooling_matrix @ true_matches).astype(bool)
    
    print(f"\n--- Query {qidx} ---")
    print(f"  #positive_pools: {np.sum(positive_pools)}, #hash_matches: {np.sum(true_matches)}")
    
    # === my_algo ===
    candidates_my = my_matching_algo(pooling_matrix.copy(), positive_pools.copy())
    # Rank candidates by hash match scores
    my_indices = np.where(candidates_my)[0]
    if len(my_indices) > 0:
        my_hash_scores = hash_match_scores[my_indices]
        sorted_idx = np.argsort(-my_hash_scores)[:K]
        topK_my = my_indices[sorted_idx]
    else:
        topK_my = np.argsort(-hash_match_scores)[:K]  # Fallback to top hash matches
    
    prec_my_hash = len(set(topK_my).intersection(set(topK_hash_indices))) / K
    rec_my_hash = len(set(topK_my).intersection(set(topK_hash_indices))) / K
    prec_my_true = len(set(topK_my).intersection(set(topK_true_indices))) / K
    rec_my_true = len(set(topK_my).intersection(set(topK_true_indices))) / K
    
    metrics["my_algo"]["precision_vs_hash"].append(prec_my_hash)
    metrics["my_algo"]["recall_vs_hash"].append(rec_my_hash)
    metrics["my_algo"]["precision_vs_true"].append(prec_my_true)
    metrics["my_algo"]["recall_vs_true"].append(rec_my_true)
    print(f"  my_algo: recovered={len(my_indices)}, P/R vs hash={prec_my_hash:.3f}/{rec_my_hash:.3f}, P/R vs true={prec_my_true:.3f}/{rec_my_true:.3f}")
    
    # === LSMR ===
    from scipy.sparse import csr_matrix
    from scipy.sparse.linalg import lsmr
    A_sparse = csr_matrix(pooling_matrix, dtype=float)
    y = positive_pools.astype(float)
    sol = lsmr(A_sparse, y)
    x_lsmr = np.nan_to_num(sol[0])
    # Get candidates above threshold
    lsmr_indices = np.where(x_lsmr > 0.5)[0]
    if len(lsmr_indices) > 0:
        lsmr_hash_scores = hash_match_scores[lsmr_indices]
        sorted_idx = np.argsort(-lsmr_hash_scores)[:K]
        topK_lsmr = lsmr_indices[sorted_idx]
    else:
        # Fallback: use x_lsmr scores directly
        topK_lsmr = np.argsort(-x_lsmr)[:K]
    
    prec_lsmr_hash = len(set(topK_lsmr).intersection(set(topK_hash_indices))) / K
    rec_lsmr_hash = len(set(topK_lsmr).intersection(set(topK_hash_indices))) / K
    prec_lsmr_true = len(set(topK_lsmr).intersection(set(topK_true_indices))) / K
    rec_lsmr_true = len(set(topK_lsmr).intersection(set(topK_true_indices))) / K
    
    metrics["lsmr"]["precision_vs_hash"].append(prec_lsmr_hash)
    metrics["lsmr"]["recall_vs_hash"].append(rec_lsmr_hash)
    metrics["lsmr"]["precision_vs_true"].append(prec_lsmr_true)
    metrics["lsmr"]["recall_vs_true"].append(rec_lsmr_true)
    print(f"  lsmr: recovered={len(lsmr_indices)}, P/R vs hash={prec_lsmr_hash:.3f}/{rec_lsmr_hash:.3f}, P/R vs true={prec_lsmr_true:.3f}/{rec_lsmr_true:.3f}")
    
    # === OMP ===
    x_omp = OMP(pooling_matrix.astype(float), positive_pools.astype(float), s=500)
    x_omp = np.nan_to_num(x_omp)
    # Get candidates above threshold
    omp_indices = np.where(x_omp > 0.5)[0]
    if len(omp_indices) > 0:
        omp_hash_scores = hash_match_scores[omp_indices]
        sorted_idx = np.argsort(-omp_hash_scores)[:K]
        topK_omp = omp_indices[sorted_idx]
    else:
        # Fallback: use x_omp scores directly
        topK_omp = np.argsort(-x_omp)[:K]
    
    prec_omp_hash = len(set(topK_omp).intersection(set(topK_hash_indices))) / K
    rec_omp_hash = len(set(topK_omp).intersection(set(topK_hash_indices))) / K
    prec_omp_true = len(set(topK_omp).intersection(set(topK_true_indices))) / K
    rec_omp_true = len(set(topK_omp).intersection(set(topK_true_indices))) / K
    
    metrics["omp"]["precision_vs_hash"].append(prec_omp_hash)
    metrics["omp"]["recall_vs_hash"].append(rec_omp_hash)
    metrics["omp"]["precision_vs_true"].append(prec_omp_true)
    metrics["omp"]["recall_vs_true"].append(rec_omp_true)
    print(f"  omp: recovered={len(omp_indices)}, P/R vs hash={prec_omp_hash:.3f}/{rec_omp_hash:.3f}, P/R vs true={prec_omp_true:.3f}/{rec_omp_true:.3f}")

# Pooling matrix diagnostics
print("\n=== Pooling Matrix Diagnostics ===")
print(f"Pooling matrix shape: {pooling_matrix.shape}")
print(f"Per-point pool memberships (min, max, mean): {np.min(np.sum(pooling_matrix, axis=0))}, {np.max(np.sum(pooling_matrix, axis=0))}, {np.mean(np.sum(pooling_matrix, axis=0)):.2f}")
print(f"Per-pool sizes (min, max, mean): {np.min(np.sum(pooling_matrix, axis=1))}, {np.max(np.sum(pooling_matrix, axis=1))}, {np.mean(np.sum(pooling_matrix, axis=1)):.2f}")

# Summary statistics
print(f"\n=== Overall Results (Top-{K}) ===")
for algo in ["my_algo", "lsmr", "omp"]:
    p_hash = np.mean(metrics[algo]["precision_vs_hash"])
    r_hash = np.mean(metrics[algo]["recall_vs_hash"])
    p_true = np.mean(metrics[algo]["precision_vs_true"])
    r_true = np.mean(metrics[algo]["recall_vs_true"])
    f1_hash = 2 * p_hash * r_hash / (p_hash + r_hash) if (p_hash + r_hash) > 0 else 0.0
    f1_true = 2 * p_true * r_true / (p_true + r_true) if (p_true + r_true) > 0 else 0.0
    print(f"{algo}:")
    print(f"  vs hash: P={p_hash:.3f}, R={r_hash:.3f}, F1={f1_hash:.3f}")
    print(f"  vs true: P={p_true:.3f}, R={r_true:.3f}, F1={f1_true:.3f}")

(5000, 2573)


Validating queries...:   0%|          | 0/10 [00:00<?, ?it/s]


--- Query 0 ---
  #positive_pools: 4641, #hash_matches: 1274
[my_matching_algo] Initial possible.sum(): 1281151 (should be 1281151)
[my_matching_algo] After negative pool elimination, possible.sum(): 608322
[my_matching_algo] Iter 0: No new identifications, breaking loop.
[my_matching_algo] Final candidates.sum(): 0
  my_algo: recovered=0, P/R vs hash=1.000/1.000, P/R vs true=0.830/0.830


  lsmr: recovered=0, P/R vs hash=0.000/0.000, P/R vs true=0.000/0.000


Validating queries...:  10%|█         | 1/10 [25:21<3:48:10, 1521.22s/it]

  omp: recovered=495, P/R vs hash=0.000/0.000, P/R vs true=0.000/0.000

--- Query 1 ---
  #positive_pools: 4608, #hash_matches: 1262
[my_matching_algo] Initial possible.sum(): 1281151 (should be 1281151)
[my_matching_algo] After negative pool elimination, possible.sum(): 565836
[my_matching_algo] Iter 0: No new identifications, breaking loop.
[my_matching_algo] Final candidates.sum(): 0
  my_algo: recovered=0, P/R vs hash=1.000/1.000, P/R vs true=0.240/0.240
  lsmr: recovered=0, P/R vs hash=0.000/0.000, P/R vs true=0.000/0.000


Validating queries...:  20%|██        | 2/10 [50:12<3:20:27, 1503.42s/it]

  omp: recovered=495, P/R vs hash=0.030/0.030, P/R vs true=0.030/0.030

--- Query 2 ---
  #positive_pools: 4602, #hash_matches: 1290
[my_matching_algo] Initial possible.sum(): 1281151 (should be 1281151)
[my_matching_algo] After negative pool elimination, possible.sum(): 558859
[my_matching_algo] Iter 0: No new identifications, breaking loop.
[my_matching_algo] Final candidates.sum(): 0
  my_algo: recovered=0, P/R vs hash=1.000/1.000, P/R vs true=0.130/0.130
  lsmr: recovered=0, P/R vs hash=0.000/0.000, P/R vs true=0.000/0.000


Validating queries...:  30%|███       | 3/10 [1:17:39<3:03:02, 1568.94s/it]

  omp: recovered=497, P/R vs hash=0.000/0.000, P/R vs true=0.000/0.000

--- Query 3 ---
  #positive_pools: 4518, #hash_matches: 1183
[my_matching_algo] Initial possible.sum(): 1281151 (should be 1281151)
[my_matching_algo] After negative pool elimination, possible.sum(): 465037
[my_matching_algo] Iter 0: No new identifications, breaking loop.
[my_matching_algo] Final candidates.sum(): 0
  my_algo: recovered=0, P/R vs hash=1.000/1.000, P/R vs true=0.150/0.150
  lsmr: recovered=0, P/R vs hash=0.000/0.000, P/R vs true=0.000/0.000


## MLGT Algorithm Quality Test Suite
This section runs systematic tests to compare the quality of `my_algo`, `lsmr`, and `lsqr` recovery algorithms. It reports precision, recall, and F1 for each method, and compares their candidate sets to hash-matching and brute-force top-K.

In [ ]:
def test_mlgt_algorithms(mlgt, query_set, num_queries=10, top_k=100):
    results = {algo: [] for algo in ["my_algo", "lsmr", "lsqr"]}
    for algo in ["my_algo", "lsmr", "lsqr"]:
        print(f"\nTesting {algo}...")
        for qidx in range(min(num_queries, query_set.shape[0])):
            query_vector = query_set[qidx]
            precision, recall, recall_saff_hash, recall_hash_true, t_saff, t_hash, t_true = mlgt.get_metrics(query_vector, algorithm=algo)
            results[algo].append({
                "precision": precision,
                "recall": recall,
                "recall_vs_hash": recall_saff_hash,
                "recall_hash_vs_true": recall_hash_true,
                "f1": 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0,
                "time": t_saff
            })
            print(f"Query {qidx}: {algo} time {t_saff:.4f}s | Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {results[algo][-1]['f1']:.4f}")
    return results

def summarize_results(results):
    import pandas as pd
    summary = {}
    for algo, res_list in results.items():
        df = pd.DataFrame(res_list)
        summary[algo] = {
            "precision_mean": df["precision"].mean(),
            "recall_mean": df["recall"].mean(),
            "f1_mean": df["f1"].mean(),
            "time_mean": df["time"].mean(),
            "recall_vs_hash_mean": df["recall_vs_hash"].mean(),
            "recall_hash_vs_true_mean": df["recall_hash_vs_true"].mean()
        }
    return pd.DataFrame(summary).T

# Run the test suite
results = test_mlgt_algorithms(mlgt, query_set, num_queries=10, top_k=100)
summarize_results(results)

### Unit Test: Recovery on Small Random y = Ax Problems
We now test the recovery algorithms on small, synthetic group testing problems (random sparse $x$, $y = Ax$) to directly evaluate their ability to solve for $x$ given $A$ and $y$.

In [ ]:
def random_group_testing_problem(num_features=50, num_pools=30, sparsity=3, pools_per_point=3, points_per_pool=5, seed=42):
    np.random.seed(seed)
    # Random binary pooling matrix
    pooling_matrix = np.zeros((num_pools, num_features), dtype=bool)
    for i in range(num_features):
        chosen = np.random.choice(num_pools, size=pools_per_point, replace=False)
        pooling_matrix[chosen, i] = 1
    for j in range(num_pools):
        if pooling_matrix[j].sum() < points_per_pool:
            # Add random points to fill pool
            missing = points_per_pool - pooling_matrix[j].sum()
            idxs = np.random.choice(np.where(~pooling_matrix[j])[0], size=missing, replace=False)
            pooling_matrix[j, idxs] = 1
    # Random sparse x
    x_true = np.zeros(num_features, dtype=bool)
    nonzero = np.random.choice(num_features, size=sparsity, replace=False)
    x_true[nonzero] = 1
    # y = Ax (binary OR)
    y = (pooling_matrix @ x_true) > 0
    return pooling_matrix, y, x_true

def test_group_testing_algos():
    pooling_matrix, y, x_true = random_group_testing_problem()
    print("True support:", np.where(x_true)[0])
    # my_algo
    from copy import deepcopy
    y_my = deepcopy(y)
    mask_my = my_matching_algo(pooling_matrix.copy(), y_my.copy())
    recovered_my = np.where(mask_my)[0]
    print("my_algo recovered:", recovered_my)
    print("my_algo recall:", np.sum(mask_my & x_true) / np.sum(x_true), "precision:", np.sum(mask_my & x_true) / max(np.sum(mask_my),1))
    # lsmr
    from scipy.sparse import csr_matrix
    y_lsmr = deepcopy(y)
    A_sparse = csr_matrix(pooling_matrix)
    from scipy.sparse.linalg import lsmr
    sol = lsmr(A_sparse, y_lsmr.astype(float))
    x_lsmr = sol[0]
    recovered_lsmr = np.where(x_lsmr > 0.5)[0]
    print("lsmr recovered:", recovered_lsmr)
    print("lsmr recall:", np.sum(x_true[recovered_lsmr]) / np.sum(x_true), "precision:", np.sum(x_true[recovered_lsmr]) / max(len(recovered_lsmr),1))
    # lsqr
    from scipy.sparse.linalg import lsqr
    sol2 = lsqr(A_sparse, y.astype(float))
    x_lsqr = sol2[0]
    recovered_lsqr = np.where(x_lsqr > 0.5)[0]
    print("lsqr recovered:", recovered_lsqr)
    print("lsqr recall:", np.sum(x_true[recovered_lsqr]) / np.sum(x_true), "precision:", np.sum(x_true[recovered_lsqr]) / max(len(recovered_lsqr),1))

test_group_testing_algos()

In [ ]:
for qidx in tqdm(range(min(num_queries, query_set.shape[0])), desc="Testing on queries..."):
    query_vector: Array1D = query_set[qidx]
    
    precision, recall, recall_saff_hash, recall_hash_true, t_saff, t_hash, t_true = mlgt.get_metrics(query_vector, algorithm=algorithm)

    print(
        f"Query {qidx}: {algorithm} time {t_saff:.4f}s | "
        f"Precision: {precision:.4f}, Recall({algorithm} vs true): {recall:.4f}, "
        f"Recall(hash vs true): {recall_hash_true:.4f}, Recall({algorithm} vs hash): {recall_saff_hash:.4f}"
    )